In [1]:
pip install transformers==4.51.3

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install sentence_transformers==2.6.1

Note: you may need to restart the kernel to use updated packages.


In [31]:
import ast
from transformers import ElectraTokenizerFast

# 1. 모델 및 Fast Tokenizer 로드
model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = ElectraTokenizerFast.from_pretrained(model_name)

# 2. 토큰 단위 레이블링 함수 정의
def get_token_labels(text, raw_label_str):
    tokens = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=128,
        padding="max_length"
    )

    offset_mapping = tokens["offset_mapping"]
    token_labels = []

    offensive_spans = []
    target_spans = []

    try:
        raw_labels = ast.literal_eval(raw_label_str)
        for ann in raw_labels:
            off_starts = ann.get("off_start_idx", [])
            off_ends = ann.get("off_end_idx", [])
            tgt_starts = ann.get("tgt_start_idx", [])
            tgt_ends = ann.get("tgt_end_idx", [])
            offensive_spans.extend(zip(off_starts, off_ends))
            target_spans.extend(zip(tgt_starts, tgt_ends))
    except Exception as e:
        print(f"Error parsing raw_label: {e}")

    for start, end in offset_mapping:
        if start is None or end is None or start == end:
            token_labels.append(0)
            continue

        is_off = any(s <= start < e for s, e in offensive_spans)
        is_tgt = any(s <= start < e for s, e in target_spans)

        if is_off and is_tgt:
            token_labels.append(3)
        elif is_off:
            token_labels.append(1)
        elif is_tgt:
            token_labels.append(2)
        else:
            token_labels.append(0)

    return tokens["input_ids"], token_labels

# 3. 예시 문장과 raw_labels (KoLD 데이터프레임에서 가져오거나 직접 입력)
text = "너는 정말 좋은 사람이야"
raw_label_str = df.iloc[0]["raw_labels"]  # 예시로 첫 번째 라벨 사용

# 4. 레이블링 수행
input_ids, labels = get_token_labels(text, raw_label_str)
tokens_decoded = tokenizer.convert_ids_to_tokens(input_ids)

# 5. 시각화 출력
print(f"{'Token':<12s} | Label")
print("-" * 25)
for token, label in zip(tokens_decoded, labels):
    if token == "[PAD]":
        break
    print(f"{token:<12s} | {label}")


TypeError: get_token_labels() missing 1 required positional argument: 'raw_label_str'

In [40]:
import pandas as pd
import ast
from tqdm import tqdm
from transformers import ElectraTokenizerFast

# 1. Load KoLD CSV
file_path = "D:\\University\\3-1\\2Text_Mining\\Team\\TM_soft_hate_classifier\\kold_v1(1).csv"
df = pd.read_csv(file_path)

# 2. Load tokenizer
tokenizer = ElectraTokenizerFast.from_pretrained("monologg/koelectra-base-v3-discriminator")

# 3. 라벨링 함수 정의
def get_token_labels(text, raw_label_str):
    tokens = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    offset_mapping = tokens["offset_mapping"]
    input_ids = tokens["input_ids"]

    labels = [0] * len(offset_mapping)

    try:
        raw_labels = ast.literal_eval(raw_label_str)
        offensive_spans = []
        target_spans = []

        for ann in raw_labels:
            offensive_spans += list(zip(ann.get("off_start_idx", []), ann.get("off_end_idx", [])))
            target_spans += list(zip(ann.get("tgt_start_idx", []), ann.get("tgt_end_idx", [])))

        # 보정 로직 예시 ("이슬람" 포함되면 target으로 간주)
        if not target_spans and "이슬람" in text:
            start = text.index("이슬람")
            end = start + len("이슬람")
            target_spans.append((start, end))

        for i, (start, end) in enumerate(offset_mapping):
            if start is None or end is None or start == end:
                continue
            is_off = any(s <= start < e for s, e in offensive_spans)
            is_tgt = any(s <= start < e for s, e in target_spans)
            if is_off and is_tgt:
                labels[i] = 3
            elif is_off:
                labels[i] = 1
            elif is_tgt:
                labels[i] = 2
    except Exception as e:
        pass

    return tokenizer.convert_ids_to_tokens(input_ids), labels

# 4. 전체 데이터 처리
token_list = []
label_list = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    tokens, labels = get_token_labels(row["comment"], row["raw_labels"])
    token_list.append(tokens)
    label_list.append(labels)

df["tokens"] = token_list
df["token_labels"] = label_list

# 5. 결과 저장
df.to_csv("kold_token_labels.csv", index=False, encoding="utf-8-sig")
print("✅ 저장 완료: kold_token_labels.csv")


100%|██████████| 40429/40429 [00:18<00:00, 2141.49it/s]


✅ 저장 완료: kold_token_labels.csv


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import ElectraForTokenClassification, ElectraTokenizerFast
import torch.optim as optim
from transformers import get_scheduler
import pandas as pd
import ast

# 1. 데이터 로드
df = pd.read_csv("kold_token_labels.csv")
tokenizer = ElectraTokenizerFast.from_pretrained("monologg/koelectra-base-v3-discriminator")

# 2. Dataset 클래스
class KoLDDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = row["comment"]
        token_labels = ast.literal_eval(row["token_labels"])

        # 재토크나이즈 (input_ids + attention_mask + offset_mapping)
        encoded = tokenizer(
            text,
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        # 길이 맞추기
        token_labels = token_labels[:128]
        token_labels += [0] * (128 - len(token_labels))

        label_masked = [label if mask == 1 else -100 for label, mask in zip(token_labels, attention_mask.tolist())]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label_masked)
        }

# 3. 학습/검증 분리
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = KoLDDataset(train_df)
val_dataset = KoLDDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# 4. 모델 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ElectraForTokenClassification.from_pretrained(
    "monologg/koelectra-base-v3-discriminator", num_labels=4
).to(device)

# 5. 옵티마이저 및 스케줄러 (AdamW 제거, torch.optim.Adam 사용)
optimizer = optim.Adam(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


Some weights of ElectraForTokenClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#6. 학습 루프
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    print(f"[Epoch {epoch+1}] Train Loss: {total_loss / len(train_loader):.4f}")

    # 검증
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            labels = batch["labels"]

            mask = labels != -100
            correct += (predictions[mask] == labels[mask]).sum().item()
            total += mask.sum().item()

    acc = correct / total if total > 0 else 0
    print(f"[Epoch {epoch+1}] Val Accuracy: {acc:.4f}")
